# Generate metric view manifest

First task of `register_metric_views`. Scans `<bundle_root>/<metric_views_dir>/<domain>/**/*.yml`, validates every file (rules MV001-MV007, errors fail the task), and sets task value `metric_view_list` for the `register_each_mv` forEach task.

Each entry: `{name, yaml_path, domain, subdomain}`, `yaml_path` relative to the bundle root, sorted by `name`.

**Widgets:** `bundle_root` (the job passes `${workspace.file_path}`), `metric_views_dir` (relative to `bundle_root`).

In [ ]:
%pip install --upgrade databricks-sdk pyyaml
dbutils.library.restartPython()

In [ ]:
dbutils.widgets.text("bundle_root", "")
dbutils.widgets.text("metric_views_dir", "")

bundle_root = dbutils.widgets.get("bundle_root").strip()
metric_views_dir = dbutils.widgets.get("metric_views_dir").strip()

missing = [k for k, v in {"bundle_root": bundle_root, "metric_views_dir": metric_views_dir}.items() if not v]
if missing:
    raise ValueError(f"Required widget(s) not set: {missing}")
print(f"bundle_root={bundle_root}\nmetric_views_dir={metric_views_dir}")

In [ ]:
import importlib
import json
import sys
import types
from pathlib import Path


def load_sdk(root: str):
    """Import mv_yaml and manifest from <root>/src/genie_ontology_modeled_knowledge_sdk_vendored/.

    The vendored files are verbatim copies of src/genie_ontology_modeled_knowledge_sdk/ (they import
    each other as genie_ontology_modeled_knowledge_sdk.*), so the vendored folder is registered as
    the genie_ontology_modeled_knowledge_sdk package. Falls back to an installed genie_ontology_modeled_knowledge_sdk.
    """
    vendored = Path(root) / "src" / "genie_ontology_modeled_knowledge_sdk_vendored"
    if (vendored / "mv_yaml.py").is_file():
        pkg = types.ModuleType("genie_ontology_modeled_knowledge_sdk")
        pkg.__path__ = [str(vendored)]
        sys.modules["genie_ontology_modeled_knowledge_sdk"] = pkg
    try:
        return (importlib.import_module("genie_ontology_modeled_knowledge_sdk.mv_yaml"),
                importlib.import_module("genie_ontology_modeled_knowledge_sdk.manifest"))
    except ImportError as exc:
        raise ImportError(
            f"genie_ontology_modeled_knowledge_sdk not found: expected the vendored library in {vendored} "
            "(scaffold_project copies it in; see src/genie_ontology_modeled_knowledge_sdk_vendored/README.md)"
        ) from exc


mv_yaml, manifest = load_sdk(bundle_root)

In [ ]:
mv_dir = Path(bundle_root) / metric_views_dir

findings = mv_yaml.validate_path(mv_dir)
errors = {str(p): [str(f) for f in fs if f.severity == "error"] for p, fs in findings.items()}
errors = {p: fs for p, fs in errors.items() if fs}
warnings = sum(1 for fs in findings.values() for f in fs if f.severity != "error")
for p, fs in errors.items():
    for f in fs:
        print(f"ERROR {p}: {f}")
if errors:
    raise ValueError(f"{sum(len(v) for v in errors.values())} metric view validation error(s) in {len(errors)} file(s)")

entries = manifest.build_manifest(mv_dir, bundle_root=bundle_root)
if not entries:
    raise ValueError(f"No metric view YAML files found under {mv_dir}/<domain>/")
problems = manifest.validate_manifest_json(entries, base_dir=bundle_root)
if problems:
    raise ValueError("Invalid manifest: " + "; ".join(problems))

for e in entries:
    print(f"  {e['name']:<40} {e['domain']}/{e['subdomain'] or '-'}  {e['yaml_path']}")
print(f"{len(entries)} metric view(s), {warnings} warning(s)")

In [ ]:
dbutils.jobs.taskValues.set(key="metric_view_list", value=entries)

summary = {"metric_views": len(entries), "warnings": warnings, "names": [e["name"] for e in entries]}
dbutils.jobs.taskValues.set(key="summary", value=summary)
dbutils.notebook.exit(json.dumps(summary))